<img src="https://hilpisch.com/tpq_logo_bic.png" width="20%" align="right">

# Python & AI in Asset Management
## Chapter 1 · Asset Management Basics and Problem Landscape

&copy; Dr. Yves J. Hilpisch<br>
AI-Powered by GPT 5.1<br>
The Python Quants GmbH | https://tpq.io<br>
https://hilpisch.com | https://linktr.ee/dyjh


### Getting Help with Python Basics
If you are new to Python, bookmark these resources while working through this notebook:

- **Chapter 3 – Python Infrastructure for Asset Management Research** explains how to install packages, manage environments, and work with notebooks step by step.
- **Appendix B – NumPy and pandas Reference** summarizes the array/dataframe idioms used below (loading CSV files, computing returns, plotting).
- **Appendix C – scikit-learn Cheat Sheet** becomes handy later when we reference ML tooling.

Refer back to those sections any time a code pattern feels unfamiliar.

## Notebook Goals

This notebook extends Chapter 1 by making the asset-management problem landscape concrete with real data, Python code, and reusable utilities. You will

- inspect the multi-asset time series in <code>data/pyaiam_eod.csv</code>,
- translate qualitative mandate components into quantitative objects,
- visualize asset-class behavior, and
- prepare lightweight tooling (returns, allocations, dashboards) that downstream chapters can reuse.

Each section ends with short exercises so you can immediately reinforce the concepts.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

plt.style.use("seaborn-v0_8")
# sns.set_context("notebook")

DATA_PATH = Path("../data/pyaiam_eod.csv")
if not DATA_PATH.exists():
    DATA_PATH = "https://hilpisch.com/pyaiam_eod.csv"

plt.rcParams.update({'font.family': 'serif', 'figure.dpi': 300})

## 1. Load and Inspect the Core Dataset

The sample file spans equities, fixed income, commodities, FX, and digital assets. Treat it as a unified research feed that can drive reporting, signal generation, and education.

_Need a refresher on pandas IO patterns? See Chapter 3 and Appendix B for a quick review of `read_csv`, indices, and time-series handling._

In [ ]:
raw = pd.read_csv(DATA_PATH, parse_dates=["Date"]).set_index("Date").sort_index()
raw.head()
# --> remove final empty rows

In [ ]:
display(raw.describe().T.assign(non_null=raw.notna().sum(), missing=raw.isna().sum()))
print(f"Date range: {raw.index.min().date()} → {raw.index.max().date()} ({len(raw):,} rows)")

## 2. Return Engineering for the Running Example

A single helper function keeps code cell clutter low and ensures consistent handling of gaps (for example, <code>EURUSD</code>).

_For vectorized NumPy/pandas math, Appendix B provides concise reminders on broadcasting and rolling operations._

In [ ]:
def prepare_return_panels(price_frame: pd.DataFrame) -> dict:
    filled = price_frame.ffill()
    log_ret = np.log(filled / filled.shift(1)).dropna()
    simple_ret = filled.pct_change().dropna()
    return {"prices": filled, "log": log_ret, "simple": simple_ret}

panels = prepare_return_panels(raw)
log_rets = panels["log"]
simple_rets = panels["simple"]
log_rets.head()

### Visualization: Cross-Asset Behavior
We compare representative equities (<code>AAPL</code>, <code>NVDA</code>), macro hedges (<code>GLD</code>, <code>TLT</code>), and diversifiers (<code>BTC-USD</code>, <code>EURUSD</code>).

_Plotting configuration tips live in Chapter 3 (Matplotlib primer) and Appendix B (visualization section)._

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(12, 10), sharex=True)
selected = ["AAPL", "NVDA", "GLD", "TLT", "BTC-USD", "EURUSD"]
(panels["prices"][selected] / panels["prices"][selected].iloc[0]).plot(ax=axes[0], linewidth=1.1, alpha=0.9)
axes[0].set_title("Indexed Price Paths (start = 1.0)")
axes[0].set_ylabel("Index Level")
log_rets[selected].rolling(63).std().mul(np.sqrt(252)).plot(ax=axes[1], linewidth=1.1, alpha=0.9)
axes[1].set_title("Rolling 3-Month Annualized Volatility")
axes[1].set_ylabel("Annualized Volatility")
# plt.tight_layout()
plt.show()

## 3. From Narrative Mandates to Data Objects
Below we connect the qualitative pieces of an investment mandate (universe, constraints, risk budgets) to concrete Python structures.

```text
Universe   → list of tickers
Constraints → dict with box/sector/factor bounds
Risk budget → target volatility or drawdown limit
```


_Translating narrative mandates into dictionaries is discussed conceptually in Chapter 1; if you would like optional Python scaffolding, revisit Chapter 3’s project-structure guidance._

In [ ]:
ASSET_CLASSES = {
    "AAPL": "US Equities",
    "NVDA": "US Equities",
    "JPM": "Financials",
    "SPY": "Multi-Asset Benchmark",
    "GLD": "Commodities",
    "TLT": "US Treasuries",
    "EURUSD": "FX",
    "BTC-USD": "Digital Assets",
}

base_universe = list(ASSET_CLASSES.keys())
constraints = {
    "long_only": True,
    "max_weight_per_asset": 0.35,
    "min_weight_per_asset": 0.0,
    "cluster_caps": {
        "US Equities": 0.6,
        "Digital Assets": 0.15,
    },
}
risk_budget = {"target_vol": 0.12, "max_drawdown": 0.2}

mandate = {
    "name": "Diversified Multi-Asset Mandate",
    "universe": base_universe,
    "constraints": constraints,
    "risk_budget": risk_budget,
}
mandate

## 4. Prototype Allocation Dashboard

We combine latest prices and realized volatilities to illustrate how research notebooks can feed portfolio construction discussions.

In [ ]:
latest_prices = panels["prices"].iloc[-1]
annualized_vol = log_rets.std() * np.sqrt(252)
sample_weights = pd.Series(1 / len(base_universe), index=base_universe, name="eq_weight")
alloc_overview = pd.DataFrame({
    "price": latest_prices,
    "annualized_vol": annualized_vol,
    "asset_class": pd.Series(ASSET_CLASSES),
    "weight": sample_weights,
}).dropna()
alloc_overview

In [ ]:
subset = alloc_overview.drop(index="BTC-USD", errors="ignore")
fig, ax = plt.subplots(figsize=(12, 5))
scatter = ax.scatter(
    subset["annualized_vol"],
    subset["price"],
    s=subset["weight"] * 3000,
    c=subset["annualized_vol"],
    cmap="viridis",
)
for idx, (asset, row) in enumerate(subset.iterrows()):
    offset = (8, 12 if idx % 2 == 0 else -14)
    ax.annotate(
        asset,
        (row["annualized_vol"], row["price"]),
        textcoords="offset points",
        xytext=offset,
        fontweight="bold",
    )
ax.set_xlabel("Annualized Volatility")
ax.set_ylabel("Latest Price")
ax.set_title("Volatility vs. Price with Sample Weights")
fig.colorbar(scatter, label="Annualized Volatility")
plt.show()

## 5. Lightweight Reporting Helpers

The function below summarizes performance diagnostics that business stakeholders expect in “at a glance” dashboards.

_Chapter 8 introduces a full backtesting/reporting stack; this notebook gives you an early glimpse, so feel free to peek ahead if you want richer context._

In [ ]:
def describe_portfolio(log_returns: pd.DataFrame, weights: pd.Series, risk_free_rate: float = 0.02):
    aligned = log_returns[weights.index].dropna()
    port_log = aligned.mul(weights).sum(axis=1)
    ann_return = port_log.mean() * 252
    ann_vol = port_log.std() * np.sqrt(252)
    sharpe = (ann_return - risk_free_rate) / ann_vol if ann_vol > 0 else np.nan
    max_dd = (np.exp(port_log.cumsum()) / np.exp(port_log.cumsum()).cummax() - 1).min()
    return pd.Series(
        {
            "annualized_return": ann_return,
            "annualized_vol": ann_vol,
            "sharpe_ratio": sharpe,
            "max_drawdown": max_dd,
        }
    )

describe_portfolio(log_rets, sample_weights)

## 6. Exercises

### Exercise 1 – Build a Cumulative-Return Function
Implement <code>plot_cumulative_return(asset, start_date=None)</code> that plots simple and log cumulative returns for any ticker.

<details>
<summary>Hint</summary>
Select the desired price series with <code>prices[asset]</code>, convert to returns with <code>.pct_change()</code> and <code>.apply(np.log1p)</code>, then use <code>.cumsum()</code>.
</details>

### Exercise 2 – Map Asset Classes to Pie-Chart Allocations
Design a function that aggregates current weights by <code>asset_class</code> and draws a pie chart suitable for mandate reviews.

<details>
<summary>Hint</summary>
Use <code>alloc_overview.groupby("asset_class")["weight"].sum()</code> prior to calling <code>plt.pie()</code>.
</details>

### Exercise 3 – Stress-Test the Constraints Dictionary
Write validation code that raises a <code>ValueError</code> when proposed weights violate <code>max_weight_per_asset</code> or <code>cluster_caps</code>.

<details>
<summary>Hint</summary>
Loop through the dictionary, compute group sums via <code>weights.groupby(asset_classes).sum()</code>, and compare to thresholds.
</details>


## 7. Takeaways for Chapter 1

- Real data makes the asset-management ecosystem tangible: time-series panels let you reason about universes, constraints, and reporting.  
- Even simple helper functions (return prep, mandate dictionaries, allocation dashboards) accelerate future chapters.  
- Embedding exercises directly inside notebooks encourages readers to adapt the tooling to their own mandates.


<img src="https://hilpisch.com/tpq_logo_bic.png" width="20%" align="right">